CONTENTS:


In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [14]:
!sudo /bin/bash -c "(source /venv/bin/activate; pip install --quiet jupyterlab-vim)"
!sudo /bin/bash -c "source /venv/bin/activate && pip install google-api-python-client"

!jupyter labextension enable

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 95.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.6/158.6 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.9/96.9 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.7/221.7 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 25.3 MB/s eta 0:00:00


In [ ]:
import logging

import pandas as pd

# /venv/lib/python3.12/site-packages/gspread_pandas/spread.py:401: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)` .replace("", np.nan)
pd.set_option("future.no_silent_downcasting", True)

import helpers.hdbg as hdbg
import helpers.henv as henv
import helpers.hprint as hprint

# hcache.get_global_cache_info()
# hcache.clear_global_cache("all")

In [3]:
hdbg.init_logger(verbosity=logging.INFO)

_LOG = logging.getLogger(__name__)

_LOG.info("%s", henv.get_system_signature()[0])

hprint.config_notebook()

INFO  > cmd='/venv/lib/python3.12/site-packages/ipykernel_launcher.py -f /home/.local/share/jupyter/runtime/kernel-2d8c1236-7801-461f-85f8-9d3549253f24.json'
INFO  # Git
  branch_name='CmampTasl11283_Scrape_airtable_into_a_CSV'
  hash='cd2804ce7'
  # Last commits:
    * cd2804ce7 CK Bot   Update helpers repo                                               (    3 days ago) Sat Jan 25 01:37:18 2025  (HEAD -> CmampTasl11283_Scrape_airtable_into_a_CSV, origin/master, origin/HEAD, master)
    * 7ed995f51 Heanh Sok CmampTask10332_Automate_sprint_management_in_GitHub_Projects (#11235) (    3 days ago) Fri Jan 24 15:50:51 2025           
    * b357ec0d4 Nina Lee CmTask11259_Perform_demand_EDA_spp_hourly_load_2 (#11272)         (    3 days ago) Fri Jan 24 12:13:49 2025           
# Machine info
  system=Linux
  node name=4860048e33b5
  release=5.15.0-1075-aws
  version=#82~20.04.1-Ubuntu SMP Thu Dec 19 05:24:09 UTC 2024
  machine=x86_64
  processor=x86_64
  cpu count=8
  cpu freq=scpufreq(current

In [15]:
from ck_marketing.process_automation.workflows import GoogleSheetsHelper as GSH

In [16]:
gsheet_helper = GSH()

In [33]:
df_air = gsheet_helper.read_sheet("10t1YxmCTu3Kl8q3QsT-CqWWuXgg28bxtrpkNUnHp8Zo")

In [34]:
df_air.head()

,Family Office,Name,Email,City,State,Website,Error_row
0,Fleming Family & Partners Limited,Ahmet Feridun,ahmet.feridun@ffandp.com,London,–,–,
1,Stamos Capital,Ron Marryott,rmarryott@stamoscapital.com,–,CA,https://www.stamoscapital.com/about/,
2,TAG Associates,John Pantowich,jpantowich@tagassoc.com,New York,NY,–,
3,HOLBEIN PARTNERS LLP,ANDERE RODGER,andere.rodger@holbeinpartners.com,London,–,http://www.holbeinpartners.com/,
4,ICONIQ Capital,Vid Mahansaria,vid@iconiqcapital.com,San Francisco,CA,http://www.iconiqcapital.com/,


In [35]:
len(df_air)

6253

In [36]:
df_air[10:15]

,Family Office,Name,Email,City,State,Website,Error_row
10,Perspecta Trust,STEPHEN J. TALL,stall@perspectatrust.com,Hampton,NH,https://perspectatrust.com/,
11,Perspecta Trust,Open,STEPHEN J. TALL,stall@perspectatrust.com,Hampton,NH,https://perspectatrust.com/
12,Habif Arogeti & Wynne LLP,Edward Deck,ed.deck@hawcpa.com,Atlanta,GA,info@hawcpa.com,
13,Piedmont Partners Group,Stephen Leist,shleist@piedmontpartnersgroup.com,Piedmont,CA,www.piedmontpartnersgroup.com,
14,Baldwin Family Office LLC,Susan Berry Kohlhas,skohlhas@baldwinim.com,West Conshohocken,PA,–,


In [41]:
def fix_shifted_rows(df):
    for index, row in df.iterrows():
        if row["Name"] == "Open":
            df.loc[index, "Name"] = row["Email"]
            df.loc[index, "Email"] = row["City"]
            df.loc[index, "City"] = row["State"]
            df.loc[index, "State"] = row["Website"]
            df.loc[index, "Website"] = row["Error_row"]
            df.loc[index, "Error_row"] = ""
    return df

In [42]:
df_air_cleaned = fix_shifted_rows(df_air)

In [43]:
len(df_air_cleaned)

6253

In [44]:
df_air_cleaned[10:15]

,Family Office,Name,Email,City,State,Website,Error_row
10,Perspecta Trust,STEPHEN J. TALL,stall@perspectatrust.com,Hampton,NH,https://perspectatrust.com/,
11,Perspecta Trust,STEPHEN J. TALL,stall@perspectatrust.com,Hampton,NH,https://perspectatrust.com/,None
12,Habif Arogeti & Wynne LLP,Edward Deck,ed.deck@hawcpa.com,Atlanta,GA,info@hawcpa.com,
13,Piedmont Partners Group,Stephen Leist,shleist@piedmontpartnersgroup.com,Piedmont,CA,www.piedmontpartnersgroup.com,
14,Baldwin Family Office LLC,Susan Berry Kohlhas,skohlhas@baldwinim.com,West Conshohocken,PA,–,


In [47]:
gsheet_helper.write_results(
    "10t1YxmCTu3Kl8q3QsT-CqWWuXgg28bxtrpkNUnHp8Zo", df_air_cleaned, "cleaned_data"
)

INFO  Results saved in the new tab: cleaned_data
